# Contextual Encoding Model

Runs Ridge Regression encoding models using XLM-RoBERTa sliding window embeddings
to predict ECoG brain activity. Mirrors Eyal's static encoding pipeline exactly,
using the same `process_embeddings` function from `static_encoding.py`.

## Conditions tested
| Condition | language_mode | en_embedding | he_embedding | ar_embedding |
|-----------|--------------|--------------|--------------|------------------|
| en | en | English (768d) | Hebrew | Arabic |
| he | he | English | Hebrew (768d) | Arabic |
| ar | ar | English | Hebrew | Arabic (768d) |
| en+he_residual | en+he | English (768d) | Hebrew residual (768d) | Arabic |
| en+ar_residual | en+ar | English (768d) | Hebrew | Arabic residual (768d) |
| noise | noise | Noise (768d) | Noise | Noise |

## Key research questions
1. Does English contextual (XLM-RoBERTa) outperform English static (FastText, Eyal)?
2. Does English + Hebrew residual outperform English alone?
3. Does English + Arabic residual outperform English alone?
4. Do all conditions outperform noise?

## Requirements
- `static_encoding.py` from Eyal's codebase (must be in the same directory)
- ECoG `.fif` files in `./subject_data/`
- Sliding window embeddings from notebook 04
- Residuals from notebook 05

## 1. Imports

In [ ]:
import sys
import os
from pathlib import Path

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / 'data').exists() and (path / 'notebooks').exists():
            return path
    raise FileNotFoundError('Could not find project root containing data/ and notebooks/')

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
STATIC_ENCODING_DIR = DATA_DIR / 'Amirim_Project_Submission' / 'Amirim_Project_Submission'
if not (STATIC_ENCODING_DIR / 'static_encoding.py').exists():
    raise FileNotFoundError(f'static_encoding.py not found in {STATIC_ENCODING_DIR}')
sys.path.insert(0, str(STATIC_ENCODING_DIR))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import io
import warnings
from contextlib import redirect_stdout

import mne
from mne_bids import BIDSPath
from nilearn.plotting import plot_markers

# Reuse Eyal's encoding pipeline exactly — no modifications
from static_encoding import process_embeddings

print('Imports ready.')
print(f'Project root        : {PROJECT_ROOT}')
print(f'static_encoding.py : {STATIC_ENCODING_DIR / "static_encoding.py"}')

## 2. Load Embeddings and Residuals

In [ ]:
WORD_LEVEL_CANDIDATES = [
    DATA_DIR / 'Amirim_Project_Submission' / 'translated_podcast_transcript_filtered.csv',
    DATA_DIR / 'Amirim_Project_Submission' / 'Amirim_Project_Submission' / 'translated_podcast_transcript_filtered.csv',
]
WORD_LEVEL_PATH = next((path for path in WORD_LEVEL_CANDIDATES if path.exists()), None)
if WORD_LEVEL_PATH is None:
    raise FileNotFoundError('Could not find translated_podcast_transcript_filtered.csv')

# Set this to choose which embeddings to use. Must match notebook 05.
#   'sliding_window'  - 1735 words, 32-word context window (notebook 04)
#   'contextual'      - 1692 words, full sentence context  (notebooks 01-03)
EMBEDDING_MODE = 'contextual'

full_transcript = pd.read_csv(WORD_LEVEL_PATH)
print(f'Embedding mode : {EMBEDDING_MODE}')

if EMBEDDING_MODE == 'sliding_window':
    E = pd.read_csv(PROCESSED_DIR / 'en_sliding_window_embeddings.csv').values.astype(np.float32)
    H = pd.read_csv(PROCESSED_DIR / 'he_sliding_window_embeddings.csv').values.astype(np.float32)
    A = pd.read_csv(PROCESSED_DIR / 'ar_sliding_window_embeddings.csv').values.astype(np.float32)
    word_level_df = full_transcript[['start', 'end', 'en', 'he', 'ar']].reset_index(drop=True)

elif EMBEDDING_MODE == 'contextual':
    en_idx = pd.read_csv(PROCESSED_DIR / 'en_contextual_matched_indices.csv')
    orig_idx = en_idx['original_word_idx'].values
    E = pd.read_csv(PROCESSED_DIR / 'en_contextual_aligned_embeddings.csv').values.astype(np.float32)
    H = pd.read_csv(PROCESSED_DIR / 'he_contextual_aligned_embeddings.csv').values[orig_idx].astype(np.float32)
    A = pd.read_csv(PROCESSED_DIR / 'ar_contextual_aligned_embeddings.csv').values[orig_idx].astype(np.float32)
    word_level_df = full_transcript.iloc[orig_idx][['start', 'end', 'en', 'he', 'ar']].reset_index(drop=True)

else:
    raise ValueError(f'Unknown EMBEDDING_MODE: {EMBEDDING_MODE!r}. Choose sliding_window or contextual.')

# Residuals must be generated by notebook 05 with the same EMBEDDING_MODE.
H_residual = np.load(PROCESSED_DIR / f'hebrew_residuals_{EMBEDDING_MODE}.npy').astype(np.float32)
A_residual = np.load(PROCESSED_DIR / f'arabic_residuals_{EMBEDDING_MODE}.npy').astype(np.float32)

# Noise control — same shape and distribution as English
rng   = np.random.RandomState(42)
NOISE = rng.normal(E.mean(), E.std(), size=E.shape).astype(np.float32)
print(f'Words            : {len(word_level_df)}')
print(f'English          : {E.shape}')
print(f'Hebrew           : {H.shape}')
print(f'Arabic           : {A.shape}')
print(f'Hebrew residual  : {H_residual.shape}')
print(f'Arabic residual  : {A_residual.shape}')
print(f'Noise control    : {NOISE.shape}')

assert E.shape == H.shape == A.shape == H_residual.shape == A_residual.shape, \
    'Shape mismatch between embedding matrices!'
print('\nAll shapes verified ✓')

## 3. Build Conditions and Base DataFrame

In [ ]:
# Base DataFrame with timing info — shared across all conditions
base_df = pd.DataFrame({
    'start': word_level_df['start'].values,
    'end'  : word_level_df['end'].values,
})

# Each condition specifies:
#   condition_name : used for output filenames
#   language_mode  : passed directly to process_embeddings
#                    'en'    -> uses en_embedding column only
#                    'he'    -> uses he_embedding column only
#                    'ar'    -> uses ar_embedding column only
#                    'en+he' -> concatenates en_embedding + he_embedding
#                    'en+ar' -> concatenates en_embedding + ar_embedding
#                    'noise' -> generates noise internally
#   en_mat, he_mat, ar_mat : what to store in each column
#
# For en+he_residual: language_mode='en+he' means process_embeddings
# concatenates en_embedding and he_embedding. We store the Hebrew residual
# in he_embedding so the result is EN (768d) + HE_RESIDUAL (768d) = 1536d.

CONDITIONS = [
    ('en',             'en',    E,      H,          A),
    ('he',             'he',    E,      H,          A),
    ('ar',             'ar',    E,      H,          A),
    ('en+he_residual', 'en+he', E,      H_residual, A),
    ('en+ar_residual', 'en+ar', E,      H,          A_residual),
    ('noise',          'noise', E,      H,          A),  # noise generated internally
]

print('Conditions to run:')
print(f'{"Name":22s}  {"Mode":10s}  {"en":12s}  {"he":12s}  {"ar"}')
print('-' * 75)
for name, mode, en, he, ar in CONDITIONS:
    print(f'{name:22s}  {mode:10s}  {str(en.shape):12s}  {str(he.shape):12s}  {ar.shape}')

## 4. Config

In [ ]:
freq       = 64
tmin, tmax = -2.0, 2.0
use_PCA    = True
PCA_dim    = 150

# Updated paths to match your local ds005574 structure
datapath = DATA_DIR / 'ds005574' / 'derivatives' / 'ecogprep'
outpath  = f'./encoding_results_{EMBEDDING_MODE}_{freq}Hz_({tmin},{tmax})/'
os.makedirs(outpath, exist_ok=True)

subjects = [f'{i:02d}' for i in range(1, 10)]

print(f'Embedding mode : {EMBEDDING_MODE}')
print(f'Frequency      : {freq} Hz')
print(f'Time window    : {tmin}s to {tmax}s')
print(f'Subjects       : {subjects}')
print(f'Conditions     : {[c[0] for c in CONDITIONS]}')
print(f'Output dir     : {outpath}')
print(f'Total runs     : {len(subjects)} x {len(CONDITIONS)} = {len(subjects)*len(CONDITIONS)}')

## 5. Encoding Loop

In [ ]:
for subj in subjects:
    # Load subject ECoG data
    file_path = BIDSPath(
        root=str(datapath),
        subject=subj, task='podcast', datatype='ieeg',
        description='highgamma', suffix='ieeg', extension='.fif'
    ).fpath  # use .fpath instead of .basename to get the full path

    fif_path = str(file_path)
    if not os.path.exists(fif_path):
        print(f'Subject {subj}: file not found at {fif_path}, skipping.')
        continue

    raw = mne.io.read_raw_fif(fif_path, verbose=False)
    print(f'\nSubject {subj}: {len(raw.info["ch_names"])} channels loaded')

    for condition_name, language_mode, en_mat, he_mat, ar_mat in tqdm(
        CONDITIONS, desc=f'Subject {subj}'
    ):
        # Build embedding_df for this condition
        # Each embedding column contains a list of float32 arrays
        # process_embeddings selects columns based on language_mode
        embedding_df = base_df.copy()
        embedding_df['en_embedding'] = list(en_mat.astype(np.float32))
        embedding_df['he_embedding'] = list(he_mat.astype(np.float32))
        embedding_df['ar_embedding'] = list(ar_mat.astype(np.float32))

        # Run Eyal's encoding pipeline with no modifications
        f = io.StringIO()
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            with redirect_stdout(f):
                _, cv_scores = process_embeddings(
                    embedding_df=embedding_df,
                    raw=raw,
                    channel_names_regex='',
                    freq=freq,
                    tmin=tmin,
                    tmax=tmax,
                    language_mode=language_mode,
                    random_noise_mode='over all embeds',
                    use_PCA=use_PCA,
                    PCA_dim=PCA_dim
                )

        # Save cv_scores
        score_fname = os.path.join(
            outpath, f'corrs subj={subj} - {condition_name}.npy'
        )
        np.save(score_fname, cv_scores)

        # Plot correlation over time
        lags = np.arange(tmin * 512, tmax * 512, (512 / freq)) / 512
        mean = cv_scores.mean((0, 1))
        err  = cv_scores.std((0, 1)) / np.sqrt(np.prod(cv_scores.shape[:2]))

        fig, ax = plt.subplots()
        ax.plot(lags, mean, color='black')
        ax.fill_between(lags, mean - err, mean + err, alpha=0.1, color='black')
        ax.axvline(0, c=(.9, .9, .9), ls='--')
        ax.axhline(0, c=(.9, .9, .9), ls='--')
        ax.set_xlabel('lag (s)')
        ax.set_ylabel('encoding performance (r ± sem)')
        ax.set_title(f'Subject {subj} — {condition_name}')
        ax.set_ylim(-0.02, 0.05)
        fig.savefig(
            os.path.join(outpath,
                f'correlation time subj={subj} - {condition_name}.png'),
            dpi=600, bbox_inches='tight'
        )
        plt.close(fig)

        # Plot correlation over electrodes
        values = cv_scores.mean(0).max(-1)
        ch2loc = {ch['ch_name']: ch['loc'][:3] for ch in raw.info['chs']}
        coords = np.vstack([ch2loc[ch] for ch in raw.info['ch_names']]) * 1000
        order  = values.argsort()

        lzr = plot_markers(
            values[order], coords[order],
            node_size=30, display_mode='lzr',
            node_vmin=0, node_vmax=0.28,
            node_cmap='inferno_r', colorbar=True
        )
        lzr.title(f'Subject {subj} — {condition_name}', size=10)
        lzr.savefig(
            os.path.join(outpath,
                f'correlation electrodes subj={subj} - {condition_name}.png'),
            dpi=600, bbox_inches='tight'
        )
        plt.close()

print('\nAll encoding runs complete.')
print(f'Results saved to: {outpath}')

## 6. Results Summary

In [ ]:
print('=' * 65)
print('ENCODING RESULTS SUMMARY — CONTEXTUAL EMBEDDINGS')
print('=' * 65)
print(f'{"Condition":22s}  {"Mean peak r":>12s}  {"Best subj r":>12s}')
print('-' * 52)

for condition_name, _, _, _, _ in CONDITIONS:
    subject_peaks = []
    for subj in subjects:
        score_fname = os.path.join(
            outpath, f'corrs subj={subj} - {condition_name}.npy'
        )
        if os.path.exists(score_fname):
            scores = np.load(score_fname)
            subject_peaks.append(scores.mean(0).max(-1).mean())

    if subject_peaks:
        mean_r = np.mean(subject_peaks)
        best_r = np.max(subject_peaks)
        print(f'{condition_name:22s}  {mean_r:12.4f}  {best_r:12.4f}')
    else:
        print(f'{condition_name:22s}  no results found')

print('\nInterpretation:')
print('  noise            should be near 0 (sanity check)')
print('  en               contextual English — compare to Eyal static baseline')
print('  en+he/ar_residual  residual conditions — do they beat English alone?')